# PGCE Counterfactual Pipeline with slimtsf

This notebook demonstrates a PGCE counterfactual workflow that starts with raw time-series CSV files, converts them into slimtsf mean-valued tabular features, and then generates counterfactuals on top of that tabular representation.

## 1) Install package and dependencies (editable)

Run this once if you opened the notebook from `examples/`.

In [ ]:
%pip install slimtsf

## 2) Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from slimtsf import IntervalStatsPoolingTransformer, SlidingWindowIntervalTransformer

from PGCE import (
    FeatureRangeConstraint,
    OrderedFeaturesConstraint,
    FeatureThresholdConstraint,
    build_constrained_explainer,
    generate_counterfactuals,
    extract_first_counterfactual_df,
    plot_counterfactual_deltas,
    plot_counterfactual_profiles,
)


## 3) Load raw time-series CSVs and convert them with slimtsf

In [ ]:
raw_csv_dir = Path("../data/raw/timeseries")
label_map_path = Path("../data/raw/label_map.csv")

def read_timeseries_csvs(csv_paths, time_column="time_tag", delimiter=","):
    frames = []
    sample_ids = []
    for path in csv_paths:
        df = pd.read_csv(path, sep=delimiter)
        if time_column in df.columns:
            df = df.sort_values(time_column).reset_index(drop=True)
        frames.append(df)
        sample_ids.append(path.name)
    numeric_sets = [set(df.select_dtypes(include=[np.number]).columns) for df in frames]
    shared_cols = sorted(set.intersection(*numeric_sets))
    if not shared_cols:
        raise ValueError("No shared numeric time-series columns were found.")
    frames = [df[shared_cols].apply(pd.to_numeric, errors="coerce").ffill().bfill() for df in frames]
    return frames, sample_ids, shared_cols

def frames_to_3d_array(frames):
    if len({frame.shape[0] for frame in frames}) != 1 or len({frame.shape[1] for frame in frames}) != 1:
        raise ValueError("All time-series frames must share the same shape.")
    return np.stack([frame.to_numpy(dtype=float).T for frame in frames], axis=0)

raw_csv_paths = sorted(raw_csv_dir.glob("*.csv"))
if not raw_csv_paths:
    raise FileNotFoundError(f"No raw time-series CSV files were found in {raw_csv_dir}.")

frames, sample_ids, source_features = read_timeseries_csvs(raw_csv_paths)
X_raw = frames_to_3d_array(frames)

stage1 = SlidingWindowIntervalTransformer(window_sizes=None, window_step_ratio=0.5, feature_functions=["mean", "std", "slope"])
interval_features = stage1.fit_transform(X_raw)
stage2 = IntervalStatsPoolingTransformer(aggregations=("min", "mean", "max"))
tabular_features = stage2.fit_transform(interval_features, feature_metadata=stage1.feature_metadata_)
feature_names = list(stage2.get_feature_names_out())
tabular_df = pd.DataFrame(tabular_features, columns=feature_names)

def load_label_mapping(label_map_path, file_column="File", label_column="Label"):
    if not label_map_path.exists():
        raise FileNotFoundError(f"Expected a label mapping CSV at {label_map_path}.")
    mapping = pd.read_csv(label_map_path)
    if file_column not in mapping.columns or label_column not in mapping.columns:
        raise ValueError(f"Label mapping CSV must contain columns: {file_column}, {label_column}")
    return mapping[[file_column, label_column]].copy()

metadata = load_label_mapping(label_map_path)
metadata = metadata.drop_duplicates(subset=["File"]).set_index("File")

labeled_df = tabular_df.copy()
labeled_df["File"] = sample_ids
labeled_df["Label"] = labeled_df["File"].map(metadata["Label"])
if labeled_df["Label"].isna().any():
    missing = labeled_df.loc[labeled_df["Label"].isna(), "File"].tolist()
    raise ValueError(f"Missing labels for files: {missing}")

labeled_df["Event_Y_N"] = pd.to_numeric(labeled_df["Label"], errors="coerce").fillna(0).astype(int)
labeled_df["Multi_Label"] = labeled_df["Label"]

train_df = labeled_df.sample(frac=0.8, random_state=42)
query_pool = labeled_df.drop(train_df.index)
train_df = train_df.reset_index(drop=True)
query_df = query_pool.drop(columns=["Label", "Event_Y_N", "Multi_Label", "File"]).iloc[[0]].reset_index(drop=True)
train_df.head()


## 4) Train a model

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(train_df.drop(columns=["target"]), train_df["target"])

## 5) Define constraints

In [ ]:
feature_names = [col for col in train_df.columns if col != "target"]
permitted_ranges = {
    col: [float(train_df[col].quantile(0.01)), float(train_df[col].quantile(0.99))]
    for col in feature_names
}

constraints = [
    FeatureRangeConstraint(ranges=permitted_ranges, penalty_value=500.0),
    OrderedFeaturesConstraint(
        ordered_feature_groups=[feature_names[:3]],
        increasing=False,
        strict=False,
        penalty_value=50.0,
    ) if len(feature_names) >= 3 else None,
    FeatureThresholdConstraint(
        feature_name=feature_names[0],
        upper_bound=float(train_df[feature_names[0]].quantile(0.95)),
        penalty_value=100.0,
    ),
]
constraints = [constraint for constraint in constraints if constraint is not None]

constraints

## 6) Build constrained explainer

In [ ]:
explainer, interfaces = build_constrained_explainer(
    dataframe=train_df,
    model=model,
    outcome_name="target",
    constraints=constraints,
    l0_penalty_weight=0.2,
)

## 7) Generate counterfactuals

Depending on your machine this can take a little while.

In [ ]:
cf_obj = generate_counterfactuals(
    explainer=explainer,
    query_instances=query_df,
    total_cfs=3,
    desired_class=1,
    maxiterations=300,
    proximity_weight=0.2,
    sparsity_weight=0.2,
    diversity_weight=2.0,
)

cf_df = extract_first_counterfactual_df(cf_obj)
cf_df

## 8) Visualize feature deltas and profiles

In [ ]:
feature_cf_df = cf_df.drop(columns=["target"], errors="ignore")
_ = plot_counterfactual_deltas(query_df.iloc[[0]], feature_cf_df, top_k=min(6, len(feature_names)))
_ = plot_counterfactual_profiles(query_df.iloc[[0]], feature_cf_df, top_k=min(6, len(feature_names)), max_counterfactuals=3)